# 01 Data Download

Mount Google Drive, install the repo, and persist raw NSE intraday and daily-context data as Parquet. The primary training path is local CSV ingestion from `BANK_NIFTY_data` / `NIFTY_data`; provider APIs are attempted only when no local data is available.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/trading_system')
REPO_DIR = BASE / 'TradingBot26'
DATA_DIR = BASE / 'data'
MODEL_DIR = BASE / 'models'
LOG_DIR = BASE / 'logs'
REPORT_DIR = BASE / 'reports'
ARTIFACT_DIR = BASE / 'artifacts'

for path in (DATA_DIR / 'raw', DATA_DIR / 'processed', MODEL_DIR, LOG_DIR, REPORT_DIR, ARTIFACT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / 'pyproject.toml').exists():
    raise FileNotFoundError(f'Copy or clone this repo to {REPO_DIR} before running this notebook')

%cd $REPO_DIR

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

In [ ]:
from dataclasses import replace
from pathlib import Path

import pandas as pd

from config import MarketConfig, PathConfig
from data.loader import MarketDataLoader, standardize_ohlcv

# Your bundled CSV folders contain index OHLC data. BANKNIFTY is used if present;
# otherwise the notebook falls back to NIFTY. Change this section if needed.
BANK_NIFTY_DIRS = [REPO_DIR / 'data' / 'BANK_NIFTY_data', DATA_DIR / 'BANK_NIFTY_data']
NIFTY_DIRS = [REPO_DIR / 'data' / 'NIFTY_data', DATA_DIR / 'NIFTY_data']

if any(path.exists() for path in BANK_NIFTY_DIRS):
    SYMBOL = 'BANKNIFTY'
    TICKER = 'BANKNIFTY'
    LOCAL_DATA_PATH = next(path for path in BANK_NIFTY_DIRS if path.exists())
elif any(path.exists() for path in NIFTY_DIRS):
    SYMBOL = 'NIFTY'
    TICKER = 'NIFTY'
    LOCAL_DATA_PATH = next(path for path in NIFTY_DIRS if path.exists())
else:
    SYMBOL = 'RELIANCE'
    TICKER = 'RELIANCE.NS'
    LOCAL_DATA_PATH = None
SERIES = 'EQ'
INTERVAL = '5m'
LOOKBACK_DAYS = 365

# Optional single-file override. Expected columns: datetime/date/index, open, high, low, close, volume.
LOCAL_INTRADAY_FILE = DATA_DIR / 'raw' / 'RELIANCE_NS_5m_source.parquet'
HAS_LOCAL_INTRADAY = LOCAL_DATA_PATH is not None or LOCAL_INTRADAY_FILE.exists()

# If no local file/folder exists, provider APIs are attempted. They are not reliable for free 5m history.
# Keep False for real training. Set True only for a short yfinance smoke test.
ALLOW_YFINANCE_5M_SMOKE = False

paths = PathConfig(
    root=BASE,
    raw_data_dir=DATA_DIR / 'raw',
    processed_data_dir=DATA_DIR / 'processed',
    artifact_dir=ARTIFACT_DIR,
    model_artifact_dir=MODEL_DIR,
    report_dir=REPORT_DIR,
)
market = MarketConfig(
    symbol=SYMBOL,
    ticker=TICKER,
    series=SERIES,
    interval=INTERVAL,
    intraday_source='local-csv' if LOCAL_DATA_PATH else ('local-file' if LOCAL_INTRADAY_FILE.exists() else 'jugaad'),
    daily_source='intraday-resample' if HAS_LOCAL_INTRADAY else 'yfinance',
    local_intraday_path=LOCAL_DATA_PATH,
    lookback_days=LOOKBACK_DAYS,
)
loader = MarketDataLoader(paths, market)

if LOCAL_INTRADAY_FILE.exists():
    if LOCAL_INTRADAY_FILE.suffix.lower() == '.csv':
        raw_intraday = pd.read_csv(LOCAL_INTRADAY_FILE)
    else:
        raw_intraday = pd.read_parquet(LOCAL_INTRADAY_FILE)
    intraday = standardize_ohlcv(raw_intraday, intraday=True, timezone=market.timezone)
    intraday.to_parquet(loader.intraday_path())
    print(f'Loaded uploaded intraday data: {loader.intraday_path()} rows={len(intraday)}')
elif LOCAL_DATA_PATH:
    result = loader.download_intraday(force_refresh=False, source='local-csv')
    print(f'Loaded local CSV intraday data: {result.path} rows={result.rows} source={result.source}')
else:
    try:
        result = loader.download_intraday(force_refresh=False)
        print(f'Downloaded intraday data: {result.path} rows={result.rows} refreshed={result.refreshed}')
    except Exception as exc:
        print(f'Primary intraday adapter failed: {exc}')
        if not ALLOW_YFINANCE_5M_SMOKE:
            raise RuntimeError('No local CSV data was found and provider APIs could not fetch 5m intraday data. Copy BANK_NIFTY_data or NIFTY_data into the repo/Drive, or enable ALLOW_YFINANCE_5M_SMOKE for a short smoke test') from exc
        smoke_market = replace(market, intraday_source='yfinance-5m', lookback_days=60)
        loader = MarketDataLoader(paths, smoke_market)
        result = loader.download_intraday(force_refresh=False, source='yfinance-5m')
        print(f'Yfinance smoke intraday data: {result.path} rows={result.rows} refreshed={result.refreshed}')

daily_result = loader.download_daily_context(force_refresh=False)
print(f'Daily context data: {daily_result.path} rows={daily_result.rows} refreshed={daily_result.refreshed}')